# Can you trust it? — validation run

Runs `scripts/validation.py` (the app's search engine, imported in-process and not modified) and shows the tables behind the "Can you trust it?" tab of `ui/fonctionnement.html`.

Run from the repository root with the server's virtual environment (`server/.venv`). The full run takes several minutes and needs the Earth tables of `pretreatment_data/`.

In [ ]:
import json, subprocess, sys
from pathlib import Path

RACINE = Path.cwd() if (Path.cwd() / 'scripts').is_dir() else Path.cwd().parent
RELANCER = False   # True: rerun the whole validation (minutes); False: read the last ui/data/validation.json
if RELANCER:
    subprocess.run([sys.executable, 'scripts/validation.py', '--prod', 'https://api.astrowrap.earth'], cwd=RACINE, check=True)
V = json.loads((RACINE / 'ui' / 'data' / 'validation.json').read_text(encoding='utf-8'))
print('run', V['generated'], 'commit', V['commit'], '(local changes)' if V['dirty'] else '')

## 1. Backtest: rank of each documented analog among all eligible Earth cells

In [ ]:
pct = lambda x: '—' if x is None else f'{100 * x:.3f}%'
print(f"{'area':<34}{'analog':<34}{'counted':>8}{'AstroWrap':>11}{'slope':>11}{'random':>9}{'gates':>7}{'top10':>7}")
for a in V['areas']:
    for e in a['analogs']:
        print(f"{a['name']:<34}{e['name'][:33]:<34}{'yes' if e['expected'] else 'no':>8}{pct(e.get('pct')):>11}"
              f"{pct(e.get('pct_slope')):>11}{pct(e['pct_random']):>9}{'pass' if e.get('passes_gates') else 'drop':>7}"
              f"{'yes' if e.get('in_top10') else 'no':>7}")
att = [e for a in V['areas'] for e in a['analogs'] if e['expected'] and e.get('rank')]
print('\nin the top 5%:', sum(e['pct'] <= 0.05 for e in att), 'of', len(att),
      '| slope alone:', sum((e.get('pct_slope') or 1) <= 0.05 for e in att),
      '| random, expected:', round(sum(e['p_top5_random'] for e in att), 1))

## 2. Stability: weights ±50% (100 runs) and repainting

In [ ]:
for s in V['stability']:
    print(f"{s['id']:<12} top-3 kept {s['top3_kept']:.0%}   all three kept in {s['all3_kept_runs']:.0%} of runs   top-10 overlap {s['top10_overlap']:.0%}")
print()
for v in V['variants']:
    print(f"{v['id']:<12}{v['variant']:<16}{v['cells']:>6} cells  {v['n_sites']:>2} sites  top-10 kept {v['top10_overlap']:.0%}")

## 3. Coverage of the top 10, and timing

In [ ]:
sites = [s for a in V['areas'] for s in a['top10']]
for k, seuil in (('urban', 0.3), ('cloud', 0.5), ('vegetation', 0.2)):
    avec = [s for s in sites if s.get(k) is not None]
    print(f"{k:<11} > {seuil:.0%}: {sum(s[k] > seuil for s in avec)} of {len(avec)} sites")
T = V['timing']
print('\nlocal engine median', T['local_median_s'], 's | live site median', T['prod_median_wall_s'], 's (wall),',
      T['prod_median_server_s'], 's (server)')